<div class="panel panel-success">
    <div class="panel-heading">
        <b> Erklärung - Das Notebook starten</b>
    </div>
    <div class="panel-body">

**Um das Notebook zu starten**:

1. Klicke oben links in der Menüleiste auf **"Cell"** und anschließend auf **"Run all"**.

**Falls ein Neustart nötig ist**:

1. Klicke oben links in der Menüleiste auf **"Kernel"** und anschließend auf **"Restart & Clear Output"**.
2. Klicke oben links in der Menüleiste auf **"Cell"** und anschließend auf **"Run all"**.

# Daten importieren

In [3]:
from treech import ClassificationTree as ct

import pandas as pd
import numpy as np

from ipywidgets import interact, interactive, fixed, interact_manual, Layout, HBox
import ipywidgets as widgets

import contextlib
import io

import plotly as py
from plotly.offline import iplot
import cufflinks as cf
cf.go_offline()

<div class="panel panel-success">
    <div class="panel-heading">
        <b> Erklärung - Daten</b>
    </div>
    <div class="panel-body">

Es wird ein Datensatz geladen, der automatisch in zwei feste Teile geteilt wird:

* **Trainingsdaten**: Damit wird ein Entscheidungsbaum erstellt.
* **Testdaten**: Damit wird überprüft, wie gut der Baum auf neuen Daten funktioniert.

Der Testdatensatz bleibt im gesamten Notebook unverändert. Den Trainingsdatensatz können wir im folgenden manipulieren.


In [5]:
DATA_FILE = 'Jim_bin_125.csv'


def read_csv():
    df = pd.read_csv(DATA_FILE, sep=';')

    # Falls der Datensatz versehentlich mit Komma statt Semikolon gelesen wurde,
    # wird er erneut mit Komma eingelesen.
    if len(df.columns) == 1:
        df = pd.read_csv(DATA_FILE, sep=',')

    # Fester 50/50-Split: Nur eine Sache soll variieren, nämlich die Verzerrung der Trainingsdaten.
    df_shuffled = df.sample(frac=1, random_state=37)
    split_index = int(len(df_shuffled) * 0.5)
    df_train = df_shuffled.iloc[:split_index].reset_index(drop=True)
    df_test = df_shuffled.iloc[split_index:].reset_index(drop=True)

    print('Gesamtdatensatz:', len(df), 'Fälle')
    print('Trainingsdaten:', len(df_train), 'Fälle')
    print('Testdaten:', len(df_test), 'Fälle')
    print('Trainingsdaten')
    display(df_train)
    print('Testdaten')
    display(df_test)
    return df_train, df_test


data_widget = interactive(read_csv)
data_widget

interactive(children=(Output(),), _dom_classes=('widget-interact',))

# Trainingsdaten verzerren

<div class="panel panel-success">
    <div class="panel-heading">
        <b> Erklärung</b>
    </div>
    <div class="panel-body">

Hier erzeugst du eine Verzerrung in den Trainingsdaten.

Du wählst:

* **Merkmal**: Welches Merkmal soll verzerrt werden?
* **Ausprägung**: Welche Gruppe soll einen bestimmten Anteil in den Trainingsdaten haben?
* **Anteil**: Wie groß soll diese Gruppe in den verzerrten Trainingsdaten ungefähr sein?

Der Testdatensatz wird dabei **nicht verändert**. Er wird später nur in Teilgruppen aufgeteilt, damit du untersuchen kannst, für welche Gruppen der Baum besser oder schlechter funktioniert.

In [8]:
def _target_share_sample(df, feature, value, target_share, random_state=0):
    """Erzeugt eine Stichprobe, in der value bei feature ungefähr target_share Anteil hat.
    Es wird nur heruntergesampelt, nicht künstlich dupliziert.
    """
    group = df[df[feature] == value]
    other = df[df[feature] != value]

    if len(group) == 0 or len(other) == 0:
        return df.copy()

    target_share = max(0.01, min(0.99, float(target_share)))
    current_share = len(group) / len(df)

    if target_share < current_share:
        # ausgewählte Ausprägung reduzieren; andere Fälle behalten
        n_group = int(round(target_share * len(other) / (1 - target_share)))
        n_group = max(1, min(len(group), n_group))
        sampled_group = group.sample(n=n_group, random_state=random_state)
        sampled_other = other
    else:
        # andere Fälle reduzieren; ausgewählte Ausprägung behalten
        n_other = int(round(len(group) * (1 - target_share) / target_share))
        n_other = max(1, min(len(other), n_other))
        sampled_group = group
        sampled_other = other.sample(n=n_other, random_state=random_state)

    biased = pd.concat([sampled_group, sampled_other], axis=0)
    biased = biased.sample(frac=1, random_state=random_state).reset_index(drop=True)
    return biased


def _distribution_table(original, biased, feature):
    orig = original[feature].value_counts(normalize=True).rename('Originale Trainingsdaten') * 100
    bias = biased[feature].value_counts(normalize=True).rename('Verzerrte Trainingsdaten') * 100
    table = pd.concat([orig, bias], axis=1).fillna(0).round(1)
    table['Differenz'] = (table['Verzerrte Trainingsdaten'] - table['Originale Trainingsdaten']).round(1)
    return table


def create_bias(Merkmal, Auspraegung, Anteil_in_Prozent):
    global biased_train_data, selected_bias_feature, selected_bias_value

    df_train = data_widget.result[0]
    selected_bias_feature = Merkmal
    selected_bias_value = Auspraegung
    target_share = Anteil_in_Prozent / 100
    biased_train_data = _target_share_sample(df_train, Merkmal, Auspraegung, target_share, random_state=0)

    print('Originale Trainingsdaten:', len(df_train), 'Fälle')
    print('Verzerrte Trainingsdaten:', len(biased_train_data), 'Fälle')
    print('\nVerteilung des ausgewählten Merkmals:')
    display(_distribution_table(df_train, biased_train_data, Merkmal))

    print('Verzerrte Trainingsdaten')
    display(biased_train_data)

    return biased_train_data


def _available_bias_features(df):
    # Bewusst kleine Auswahl: Die Lernenden steuern nur die Verzerrung, nicht die Modellparameter.
    candidates = [
        'Schulform_gruppiert',
        'Geschlecht',
        'Twitch_nutzen',
        'Instagram_nutzen',
        'Alter_gruppiert'
    ]
    return [c for c in candidates if c in df.columns]


def _options_for_feature(feature):
    values = data_widget.result[0][feature].dropna().unique()
    values = sorted(values, key=lambda x: str(x))
    return values


biased_train_data = data_widget.result[0].copy()
selected_bias_feature = None
selected_bias_value = None

bias_features = _available_bias_features(data_widget.result[0])

merkmal_dropdown = widgets.Dropdown(
    options=bias_features,
    description='Merkmal'
)

auspraegung_dropdown = widgets.Dropdown(
    options=_options_for_feature(merkmal_dropdown.value),
    description='Ausprägung'
)


def _update_auspraegung_options(change):
    selected_feature = change['new']
    new_options = _options_for_feature(selected_feature)
    auspraegung_dropdown.options = new_options
    if len(new_options) > 0:
        auspraegung_dropdown.value = new_options[0]


merkmal_dropdown.observe(_update_auspraegung_options, names='value')

bias_widget = interactive(
    create_bias,
    {'manual': True, 'manual_name': 'Verzerrte Trainingsdaten erzeugen'},
    Merkmal=merkmal_dropdown,
    Auspraegung=auspraegung_dropdown,
    Anteil_in_Prozent=widgets.IntSlider(
        value=50, min=1, max=99, step=1, description='Anteil %'
    )
)

bias_widget

interactive(children=(Dropdown(description='Merkmal', options=('Schulform_gruppiert', 'Geschlecht', 'Twitch_nu…

# Entscheidungsbäume trainieren und vergleichen

## Vergleich: ursprüngliche und verzerrte Trainingsdaten

<div class="panel panel-success">
    <div class="panel-heading">
        <b> Erklärung - Zwei Bäume vergleichen</b>
    </div>
    <div class="panel-body">

Hier werden zwei Entscheidungsbäume automatisch erstellt und gestutzt:

* ein Baum mit den ursprünglichen Trainingsdaten
* ein Baum mit den verzerrten Trainingsdaten

Im Hintegrund passiert für beide Datensätze folgendes:

1. Kurz vor dem Training werden die aktuellen Trainingsdaten nochmal in Trainings- und Validierungsdaten aufgeteilt.
2. Zuerst wird aus den Trainingsdaten ein großer Entscheidungsbaum erstellt.
3. Danach wird der Baum mit den Validierungsdaten automatisch gestutzt.
4. Anschließend wird untersucht, wie gut der Baum auf den Testdaten funktioniert.

Beide Bäume verwenden dieselbe ausgewählte Zielvariable und werden anschließend auf demselben Testdatensatz bewertet.

In [12]:
def _accuracy_table(tree, train_data, test_data, bias_feature):
    rows = []

    rows.append({
        'Datensatz': 'Trainingsdaten',
        'Fälle': len(train_data),
        'Rate korrekter Klassifikationen': round(tree.prediction_accuracy(train_data) * 100, 1)
    })

    rows.append({
        'Datensatz': 'Testdaten gesamt',
        'Fälle': len(test_data),
        'Rate korrekter Klassifikationen': round(tree.prediction_accuracy(test_data) * 100, 1)
    })

    if bias_feature is not None and bias_feature in test_data.columns:
        for value in sorted(test_data[bias_feature].dropna().unique(), key=lambda x: str(x)):
            subgroup = test_data[test_data[bias_feature] == value]
            if len(subgroup) > 0:
                rows.append({
                    'Datensatz': f'Testdaten: {bias_feature} = {value}',
                    'Fälle': len(subgroup),
                    'Rate korrekter Klassifikationen': round(tree.prediction_accuracy(subgroup) * 100, 1)
                })

    return pd.DataFrame(rows)


def _split_training_validation(train_data, validation_share=0.34, random_state=1):
    """Teilt die aktuell ausgewählten Trainingsdaten intern in Konstruktions- und Validierungsdaten."""
    validation_data = train_data.sample(frac=validation_share, random_state=random_state)
    construction_data = train_data.drop(validation_data.index)
    return construction_data.reset_index(drop=True), validation_data.reset_index(drop=True)


def _available_target_features(df):
    """Mögliche Zielvariablen: kategoriale Variablen mit mindestens zwei Ausprägungen."""
    excluded = {'Klassenstufe', 'Alter'}
    candidates = []
    for column in df.columns:
        if column in excluded:
            continue
        values = df[column].dropna().unique()
        if len(values) >= 2 and df[column].dtype == 'object':
            candidates.append(column)
    return sorted(candidates, key=lambda x: str(x).lower())


def _fit_pruned_tree(train_data, target):
    """Erstellt zunächst einen großen Baum und stutzt ihn dann mit validation_pruning()."""
    construction_data, validation_data = _split_training_validation(train_data)

    tree = ct.DecisionTree(data=construction_data, target=target, crit='misclassification_error')

    # Keine zusätzlichen Termination Criteria: Der Baum wächst bis zu den Standard-Stoppregeln.
    tree.grow_tree(crit='entropy')

    # validation_pruning() gibt viele interne Statusmeldungen aus. Diese werden für die Lernenden ausgeblendet.
    with contextlib.redirect_stdout(io.StringIO()):
        tree.validation_pruning(validation_data)

    return tree


def grow_tree(Zielvariable):
    original_train = data_widget.result[0]
    test_data = data_widget.result[1]
    train_data = biased_train_data

    tree = _fit_pruned_tree(train_data, Zielvariable)

    print('Zielvariable:', Zielvariable)
    print('Automatisch gestutzter Baum mit verzerrten Trainingsdaten')
    tree.print_tree()
    display(tree.tree_graph)

    print('Rate der korrekten Klassifikationen:')
    display(_accuracy_table(tree, train_data, test_data, selected_bias_feature))

    return tree


target_dropdown = widgets.Dropdown(
    options=_available_target_features(data_widget.result[0]),
    value='spielen_onlinespiele' if 'spielen_onlinespiele' in data_widget.result[0].columns else _available_target_features(data_widget.result[0])[0],
    description='Zielvariable'
)

def compare_trees(Zielvariable):
    original_train = data_widget.result[0]
    test_data = data_widget.result[1]

    tree_original = _fit_pruned_tree(original_train, Zielvariable)
    tree_biased = _fit_pruned_tree(biased_train_data, Zielvariable)

    print('Zielvariable:', Zielvariable)
    print('Baum mit ursprünglichen Trainingsdaten')
    tree_original.print_tree()
    display(tree_original.tree_graph)
    display(_accuracy_table(tree_original, original_train, test_data, selected_bias_feature))

    print('Baum mit verzerrten Trainingsdaten')
    tree_biased.print_tree()
    display(tree_biased.tree_graph)
    display(_accuracy_table(tree_biased, biased_train_data, test_data, selected_bias_feature))

    return tree_original, tree_biased


compare_widget = interactive(
    compare_trees,
    {'manual': True, 'manual_name': 'Bäume vergleichen'},
    Zielvariable=target_dropdown
)

compare_widget

interactive(children=(Dropdown(description='Zielvariable', options=('ActionCam_YouTube', 'Adventure_spielen', …

## Entscheidungsbaum ausgeben

In [14]:
def output_tree():
    tree_widget.result.tree_graph.view()

output_tree_widget = interactive(
    output_tree,
    {'manual': True, 'manual_name': 'Tree ausgeben'}
)
output_tree_widget

interactive(children=(Button(description='Tree ausgeben', style=ButtonStyle()), Output()), _dom_classes=('widg…